In [1]:
%matplotlib inline

### Подключаем модули

In [2]:
import clickhouse_connect
import pandas as pd
import pandas_summary as ps
import matplotlib.pyplot as plt
from data_profiling import ProfileReport
import seaborn as sns

### Устанавливаем соединение с Clickhouse

In [3]:
client = clickhouse_connect.get_client(
    host='service.db_clickhouse',
    port=8123,
    username='default',
    password=''
)

### Получаем общее кол-во записей в таблице

In [4]:
sql = """SELECT COUNT(*) FROM "cars"."car_sales";"""
result = client.query(sql)
rows_count = result.result_rows[0][0]
sample_size = round(rows_count * 0.001)
print(f"Всего записей = {rows_count}, Выбока = {sample_size}")

Всего записей = 1294757, Выбока = 1295


### Формируем выборку из `sample_size` записей а также конвертирует поля в необходимый формат

In [5]:
sql = f"""SELECT * FROM "cars"."car_sales" ORDER BY rand() LIMIT {sample_size};"""
result = client.query(sql)
df = pd.DataFrame(result.result_rows, columns=result.column_names)

df['price'] = pd.to_numeric(df['price'], errors='coerce')
df['year'] = pd.to_numeric(df['year'], errors='coerce', downcast='integer')

df = df.drop(columns=['link', 'description'])

date_columns = ['date', 'parse_date']

for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], format='%Y-%m-%d', errors='coerce')

In [6]:
df

,brand,name,bodyType,color,fuelType,year,mileage,transmission,power,price,vehicleConfiguration,engineName,engineDisplacement,date,location,parse_date
0,Лада,2113 Самара,Хэтчбек 3 дв.,Черный,Бензин,2007.0,99000.0,Механика,80.0,175000,1.6 MT 21134-20-010,ВАЗ-21114,1.6,2023-05-30,Челябинск,2023-05-30 08:00:00
1,Opel,Astra GTC,Хэтчбек 3 дв.,Серый,Бензин,2007.0,248000.0,Механика,115.0,530000,1.6 MT Enjoy,Z16XER,1.6,2023-05-26,Москва,2023-05-27 00:00:00
2,Renault,Kaptur,Джип 5 дв.,Белый,Бензин,2019.0,120000.0,АКПП,143.0,1480000,2.0 AT 4WD Drive,F4R,2.0,2023-05-22,Обнинск,2023-05-25 06:00:00
3,Лада,Ларгус,Универсал,Белый,Бензин,NaN,88000.0,Механика,87.0,820000,None,None,NaN,2023-05-14,Стерлитамак,2023-05-16 11:00:00
4,УАЗ,Патриот,Джип 5 дв.,None,Бензин,2020.0,NaN,АКПП,150.0,2268000,2.7 AT Люкс Премиум,ЗМЗ-409051 ZMZ Pro,2.7,2023-05-15,Москва,2023-05-15 22:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1290,Лада,4x4 2121 Нива,Джип 3 дв.,Зеленый,Бензин,NaN,NaN,Механика,83.0,343000,None,None,NaN,2023-06-11,Москва,2023-06-11 18:00:00
1291,Ford,Focus,Хэтчбек 5 дв.,Оранжевый,Бензин,2005.0,188000.0,АКПП,100.0,360000,1.6 AT Ghia,"SHDC, HWDA, HWDB, SHDA, SHDB",1.6,2023-05-30,Санкт-Петербург,2023-05-31 07:00:00
1292,Toyota,Platz,Седан,Розовый,Бензин,1999.0,NaN,АКПП,70.0,250000,1.0 F,1SZ-FE,1.0,2023-06-21,Чунский,2023-06-21 17:00:00
1293,Daihatsu,Rocky,Джип 3 дв.,Серый,Бензин,1993.0,200000.0,Механика,105.0,365000,1.6 SE,HD-E,1.6,2023-06-06,Ванино,2023-06-06 22:00:00


In [7]:
df.dtypes

brand                           object
name                            object
bodyType                        object
color                           object
fuelType                        object
year                           float64
mileage                        float64
transmission                    object
power                          float64
price                            int64
vehicleConfiguration            object
engineName                      object
engineDisplacement             float64
date                    datetime64[ns]
location                        object
parse_date              datetime64[ns]
dtype: object

In [8]:
dfs = ps.DataFrameSummary(df)
print('categoricals: ', dfs.categoricals.tolist())
print('numerics:', dfs.numerics.tolist())

dfs.summary()

ValueError: could not convert string to float: 'Лада'

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 10))

# Заполняем сетку графиками
sns.regplot(data=df, x='price', y='power', ax=axes[0, 0])
sns.regplot(data=df, x='price', y='mileage', ax=axes[0, 1])
sns.regplot(data=df, x='price', y='year', ax=axes[0, 2])
sns.regplot(data=df, x='power', y='engineDisplacement', ax=axes[1, 0])
sns.regplot(data=df, x='year', y='power', ax=axes[1, 1])
sns.regplot(data=df, x='year', y='mileage', ax=axes[1, 2])
sns.regplot(data=df, x='price', y='engineDisplacement', ax=axes[2, 0])
sns.regplot(data=df, x='year', y='engineDisplacement', ax=axes[2, 1])

# Настраиваем внешний вид
# plt.tight_layout()
plt.show()

In [ ]:
# df = sns.load_dataset("penguins")

g = sns.PairGrid(df, diag_sharey=False)
g.map_upper(sns.scatterplot, s=15)
g.map_lower(sns.kdeplot)
g.map_diag(sns.kdeplot, lw=2)

In [ ]:
plt.figure(figsize=(16, 6))
sns.lineplot(x='date', y='price', data=df, linewidth=2.5, errorbar='sd')
# sns.lineplot(x='date', y='price', data=df, linewidth=2.5, errorbar=None)
plt.xlabel('Дата') 
plt.ylabel('Цена') 
plt.show()

In [ ]:
df['year_month'] = df['date'].dt.to_period('M')
df1 = df.sort_values(by='year_month')
plt.figure(figsize=(16, 6))
sns.boxplot(x=df['year_month'], y="price", data=df1)
sns.despine(offset=10, trim=True)
plt.xticks(rotation=90)
plt.show()

### Формируем `ProfileReport` отчет

In [ ]:
correlations_config = {
    "pearson": {"calculate": True},
    "spearman": {"calculate": True},
    "kendall": {"calculate": True},
    "phi_k": {"calculate": True},
    "cramers": {"calculate": True}
}
missing_config = {
    "bar": True,      # столбчатая диаграмма
    "matrix": True,   # матрица пропусков
    "heatmap": True, # тепловая карта
    "dendrogram": True # дендрограмма
}
profile = ProfileReport(
    df, 
    missing_diagrams=missing_config, 
    correlations=correlations_config, 
    explorative=True, 
    title='Отчёт по данным')
profile.to_file('отчёт_по_данным.html')